# Task 8 — RAG-Powered Talent Search Engine

## 1. Setup


In [21]:
%pip install -q langchain langchain-community chromadb sentence-transformers \
    transformers torch openai tiktoken pandas


In [22]:
import os
import sys
sys.path.insert(0, ".")

import pandas as pd
import rag_utils as ru

# Optional: set this to use OpenAI for the LLM-based evaluation step instead of the
# local HuggingFace fallback. Leave unset to use the free local model / rule-based fallback.
# os.environ["OPENAI_API_KEY"] = "sk-..."

DATASET_PATH = "Entity_Recognition_in_Resumes.json"  # <- matches the filename as uploaded here.
# load_resumes() also auto-detects common filename variants (spaces/underscores/hyphens)
# and falls back to a "data/" subfolder or a glob match, so this stays robust even if you
# rename or re-download the file -- but if loading ever prints a "WARNING: real dataset
# not found" message, it means the 220-resume corpus was NOT used and you fell back to
# the 12-sample synthetic demo data.


## 2. Load & parse resumes


In [23]:
resumes = ru.load_resumes(DATASET_PATH)

df = pd.DataFrame([{
    "id": r.id, "name": r.name, "category": r.category,
    "experience_level": r.experience_level,
    "num_skills": len(r.skills), "skills": ", ".join(r.skills[:8]),
    "resume_length_chars": len(r.text),
} for r in resumes])

print(f"Loaded {len(resumes)} resumes")
df.head(10)


Loaded 220 resumes from 'Entity Recognition in Resumes.json'.
Loaded 220 resumes


,id,name,category,experience_level,num_skills,skills,resume_length_chars
0,resume_0,Abhishek Jha,B.E in Information science and engineering,Junior,16,"c++, computer networks, database, database man...",1622
1,resume_1,Afreen Jamadar,General,Junior,18,"access, c++, database, database: ms access, ht...",1240
2,resume_2,Akhil Yadav Polemaina,Senior Systems Engineer,Senior,5,"cobol, jcl, mainframe, servicenow, teradata",4264
3,resume_3,Alok Khandai,Operational Analyst (SQL DBA) Engineer,Senior,14,"business, c++, database, development studio, p...",8384
4,resume_4,Ananya Chavan,lecturer,Junior,28,"10, access, ajax &jquery, ajax (less, angular ...",2933
5,resume_5,Anvitha Rao,Automation developer,Junior,11,"c++, d3js, gephi, hadoop and spark, html/css, ...",2931
6,resume_6,arjun ks,Process Specialist,Senior,1,pmp trained six sigma yellow belt,5796
7,resume_7,Arun Elumalai,QA Tester,Senior,19,"ansys, automation testing, catia, catia v6, cr...",2083
8,resume_8,Candidate 9,General,Senior,2,"doeacc o level, m.s. office",1721
9,resume_9,Ashok Kunam,Team Lead,Senior,47,"apache, apache nifi, app/web servers: tomcat, ...",4960


In [24]:
# Quick EDA: what categories / experience levels are in the corpus?
display(df["experience_level"].value_counts())
display(df["category"].value_counts().head(15))


,count
experience_level,
Senior,127
Junior,50
Mid,43


,count
category,
General,19
Software Engineer,6
Associate Consultant,4
Technical Support Engineer,3
Systems Engineer,3
Developer,3
Test Engineer,3
Technology Analyst,3
Senior Systems Engineer,3


## 3. Embed resumes & build the ChromaDB vector store



In [25]:
import importlib
import rag_utils
importlib.reload(rag_utils)
ru = rag_utils
print(ru.__file__)

vectorstore = ru.build_vector_store(resumes, persist_dir=ru.CHROMA_PERSIST_DIR)
print("Vector store built with", vectorstore._collection.count(), "resumes.")


/content/rag_utils.py


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built with 220 resumes.


## 4. Semantic search



In [26]:
query = "Junior Data Analyst who knows SQL and Tableau"
top_candidates = ru.semantic_search(vectorstore, query, k=3)

pd.DataFrame(top_candidates)[["name", "category", "experience_level", "score", "skills"]]


,name,category,experience_level,score,skills
0,Alok Khandai,Operational Analyst (SQL DBA) Engineer,Senior,0.4235,"business, c++, database, development studio, p..."
1,Meenalochani Kondya,Systems Engineer,Senior,0.3394,"ajax, asp, asp.net, css 3.0, engineer, ide ms ..."
2,Rupesh Reddy,Technology Consultant,Senior,0.3084,"articulation skills, cross cultural sensitivit..."


## 5. LLM-based evaluation (Industry Constraint)


In [27]:
llm = ru.get_llm()
evaluated = ru.evaluate_top_candidates(query, top_candidates, llm=llm)

for c in evaluated:
    print(f"\n### {c['name']}  —  {c['experience_level']} {c['category']}  (score={c['score']})")
    print(c["llm_summary"])


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Using local HuggingFace model (flan-t5-base) for LLM-based evaluation.

### Alok Khandai  —  Senior Operational Analyst (SQL DBA) Engineer  (score=0.4235)
Alok Khandai is an Operational Analyst who knows SQL and Tableau. He has 3.5 years of IT experience in SQL Database Administration, System Analysis, Design, Development & Support of MS SQL Servers in Production, Development environments & Replication and Cluster Server Environments.

### Meenalochani Kondya  —  Senior Systems Engineer  (score=0.3394)
Meenalochani Kondya (Senior Systems Engineer) is a plausible match for the query "Junior Data Analyst who knows SQL and Tableau". Matched keywords: data, sql. Listed skills: ajax, asp, asp.net, css 3.0, engineer, ide ms visual studio 2015, javascript, jquery, languages c#, management, microsoft office ms powerpoint, ms asp, ms excel, ms visio, ms word, mvc 4.0, net, operating systems microsoft windows, server 2012, source code tfs 2013, sql, sql server 2012, vss 2012, web authoring tools

## 6. Bonus — Bias Check


In [28]:
report = ru.bias_check(top_candidates, resumes)
print(ru.format_bias_report(report))


## Bias Check Report

**Category distribution (corpus vs retrieved), for categories present in the results:**
- Systems Engineer: corpus 1% -> retrieved 33%
- Technology Consultant: corpus 0% -> retrieved 33%
- Operational Analyst (SQL DBA) Engineer: corpus 0% -> retrieved 33%
- _(+167 other categories in the corpus, 0% of this result set -- omitted here for readability)_

**Avg resume length:** corpus 3697.1 chars vs retrieved 6475.0 chars

**Illustrative gender-name-hint distribution:**
- Corpus: {'Unknown': 1.0}
- Retrieved: {'Unknown': 1.0}
- _Gender hints are inferred from a tiny hard-coded first-name list for demonstration only. They are unreliable and should never be used for real hiring decisions -- use consented, validated demographic data and a proper fairness library (e.g. Fairlearn, AIF360) in production._

**Flags:**
- Retrieved resumes are, on average, notably longer than the corpus average -- possible bias toward verbose resumes rather than relevance.


## 7. Putting it together: one function = the whole search engine


In [29]:
def talent_search(query: str, k: int = 3, run_bias_check: bool = True):
    top = ru.semantic_search(vectorstore, query, k=k)
    evaluated = ru.evaluate_top_candidates(query, top, llm=llm)
    bias_report = ru.bias_check(top, resumes) if run_bias_check else None
    return evaluated, bias_report

for test_query in [
    "Senior Data Scientist with leadership experience and Python skills",
    "Entry level software engineer familiar with Java and React",
]:
    print("="*100)
    print("QUERY:", test_query)
    results, bias = talent_search(test_query, k=3)
    for c in results:
        print(f"\n- {c['name']} ({c['experience_level']} {c['category']}, score={c['score']})")
        print(f"  {c['llm_summary']}")
    print("\n" + ru.format_bias_report(bias))


QUERY: Senior Data Scientist with leadership experience and Python skills

- Yasothai Jayaramachandran (Senior Lead Engineer, score=0.2837)
  qualms you may have about this person's qualifications or experience. If you have any questions, please don't hesitate to contact me.

- Shreya Agnihotri (Senior Senior System Engineer at Infosys Limited, score=0.2403)
  Shreya Agnihotri (Senior Senior System Engineer at Infosys Limited) is a plausible match for the query "Senior Data Scientist with leadership experience and Python skills". Matched keywords: experience, python, senior, skills. Listed skills: ajax, apache kafka, databases: sql, docker, grafana, html5, java, javascript, kafka, kibana, operating systems: linux, programming languages python, prometheus, scripting language jquery, sql, technical profile:, tools and utilities: elasticsearch, web technologies python. [Generated by rule-based fallback -- configure OPENAI_API_KEY or a local HF model for a richer LLM-written explanation.]


## 8. Chat-style follow-up questions


In [30]:
candidate = top_candidates[0]
followup = "Does this candidate have leadership experience?"
answer = ru.answer_question_about_candidate(followup, candidate, llm=llm)

print(f"Candidate: {candidate['name']}")
print(f"Q: {followup}")
print(f"A: {answer}")


Candidate: Alok Khandai
Q: Does this candidate have leadership experience?
A: SQL Server security and Object permissions like maintaining Database authentication modes, creation of users, configuring permissions and assigning roles to users.  Experience in creating Jobs, Alerts, SQL Mail Agent
